# Đề xuất đặc trưng và bảng nhóm biến

**Dương Hoàng Anh — MSSV 0364169.**

Notebook nháp minh họa **9 đặc trưng cốt lõi** bằng công thức pandas.
Bảng 79 biến gốc và 21 ý tưởng đề xuất nằm trong `01_Report/HoangAnh/`;
12 ý tưởng thử thêm được giữ trong tài liệu để nhóm cân nhắc sau.

Dùng `load_split()` chung của nhóm, chỉ phân tích `X_train`. Không fit mô hình hoặc đánh giá holdout.
Chọn kernel **Python 3** trỏ tới `.venv`; repo hiện thống nhất **Python 3.11**.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Notebook nằm trong notebooks/drafts/, cách 03_SourceCode hai cấp.
sys.path.insert(0, str(Path.cwd().parents[1]))

from src.data import load_split
from src.feature_proposal import REPORT_DIR, preview_features

X_train, _, _, _ = load_split()
print('Python:', sys.version.split()[0])
print('Dữ liệu train:', X_train.shape)

Python: 3.11.9
Dữ liệu train: (1168, 79)


## 1 Bảng nhóm biến

Bảng CSV có đủ 79 biến thuộc 12 nhóm, kèm ý nghĩa, hướng tiền xử lý và số giá trị thiếu.
`MSSubClass` là mã phân loại; `MoSold` là tháng; `OverallQual` và `OverallCond` là điểm thứ bậc.
Xem toàn bộ từng biến trong `Bang_nhom_bien.csv`.

In [2]:
groups = pd.read_csv(REPORT_DIR / 'Bang_nhom_bien.csv', keep_default_na=False)
display(groups.groupby('nhom', sort=False).size().to_frame('Số biến'))
display(groups[['bien', 'nhom', 'y_nghia']].head(10))

,Số biến
nhom,
Loại hình nhà,3
Vị trí và lô đất,12
Hệ thống và tiện nghi trong nhà,8
Chất lượng và kết cấu,11
Thời gian,5
Tầng hầm,11
Diện tích và phòng trên mặt đất,8
Phòng tắm trên mặt đất,2
Garage và lối xe,7


,bien,nhom,y_nghia
0,MSSubClass,Loại hình nhà,"Mã loại nhà; số là nhãn, không phải đại lượng ..."
1,MSZoning,Vị trí và lô đất,Phân vùng sử dụng đất
2,LotFrontage,Vị trí và lô đất,"Chiều dài mặt tiền tiếp giáp đường, feet"
3,LotArea,Vị trí và lô đất,"Diện tích lô đất, ft²"
4,Street,Vị trí và lô đất,Loại mặt đường
5,Alley,Vị trí và lô đất,Loại đường ngõ; NA theo mô tả là không có ngõ
6,LotShape,Vị trí và lô đất,Hình dạng lô đất
7,LandContour,Vị trí và lô đất,Địa hình lô đất
8,Utilities,Hệ thống và tiện nghi trong nhà,Các dịch vụ hạ tầng được cung cấp
9,LotConfig,Vị trí và lô đất,Vị trí và cách bố trí lô đất


## 2 Chín công thức cốt lõi

Code ở `src/feature_proposal.py` chỉ dùng phép cộng, phép trừ và so sánh.
`TotalSF` không cộng thêm `GrLivArea` để tránh đếm trùng. Tuổi tính theo năm bán từng dòng.
`TotalBath` dùng trọng số 0,5 cho half-bath như một quy ước.

In [3]:
proposals = pd.read_csv(REPORT_DIR / 'Bang_dac_trung_de_xuat.csv')
core = proposals[proposals['uu_tien'] == 'Cốt lõi']
with pd.option_context('display.max_colwidth', 100):
    display(core[['dac_trung', 'cong_thuc', 'y_nghia_va_gia_thuyet']])

,dac_trung,cong_thuc,y_nghia_va_gia_thuyet
0,TotalSF,TotalBsmtSF + 1stFlrSF + 2ndFlrSF,Tổng quy mô tầng hầm và hai tầng chính; gồm hầm chưa hoàn thiện
1,HouseAge,YrSold - YearBuilt,Tuổi nhà tại năm bán; có thể phản ánh hao mòn
2,RemodelAge,YrSold - YearRemodAdd,Thời gian từ lần cải tạo đến năm bán
3,TotalBath,FullBath + BsmtFullBath + 0.5 * (HalfBath + BsmtHalfBath),Gộp tiện nghi tắm ở cả trên và dưới đất
4,TotalOutdoorSF,WoodDeckSF + OpenPorchSF + EnclosedPorch + 3SsnPorch + ScreenPorch,Quy mô sàn gỗ và các loại hiên; không gồm hồ bơi
5,HasGarage,1 nếu GarageCars > 0 hoặc GarageArea > 0; 0 nếu cả hai = 0,Tách trạng thái có garage khỏi quy mô garage
6,HasBsmt,1 nếu TotalBsmtSF > 0; 0 nếu = 0,Phân biệt nhà có tầng hầm và không có tầng hầm
7,HasFireplace,1 nếu Fireplaces > 0; 0 nếu = 0,Biểu diễn trạng thái có hoặc không có lò sưởi
8,HasRemodel,1 nếu YearRemodAdd > YearBuilt; 0 nếu bằng nhau,Phân biệt nhà đã cải tạo với nhà chưa cải tạo


## 3 Minh họa trên tập train

Bảng dưới dùng 1.168 căn train trước khi lọc ngoại lai.
Chưa impute hoặc chuẩn hóa; Hinh sẽ đưa các công thức được chọn vào Pipeline sau khi nhóm thống nhất.

In [4]:
features = preview_features(X_train)
display(features.head())
display(features.describe().T[['min', '50%', 'max']])
display(features.isna().sum().to_frame('Số giá trị thiếu'))

,TotalSF,HouseAge,RemodelAge,TotalBath,TotalOutdoorSF,HasGarage,HasBsmt,HasFireplace,HasRemodel
0,2566,5,5.0,3.5,61,1,1,0,0.0
1,2524,31,31.0,2.5,298,1,1,1,0.0
2,2706,7,6.0,3.5,42,1,1,1,1.0
3,2473,91,36.0,2.0,307,1,1,1,1.0
4,3343,8,8.0,3.5,276,1,1,1,0.0


,min,50%,max
TotalSF,334.0,2494.0,11752.0
HouseAge,0.0,35.0,136.0
RemodelAge,0.0,14.0,60.0
TotalBath,1.0,2.0,6.0
TotalOutdoorSF,0.0,168.0,1027.0
HasGarage,0.0,1.0,1.0
HasBsmt,0.0,1.0,1.0
HasFireplace,0.0,1.0,1.0
HasRemodel,0.0,0.0,1.0


,Số giá trị thiếu
TotalSF,0
HouseAge,0
RemodelAge,1
TotalBath,0
TotalOutdoorSF,0
HasGarage,0
HasBsmt,0
HasFireplace,0
HasRemodel,1


## 4 Một trường hợp cần đối chiếu

Train có căn Id 524 ghi năm cải tạo 2008 nhưng năm bán 2007.
Code giữ `RemodelAge` và `HasRemodel` là NA ở dòng này, không sửa năm nguồn.
Đây là ngoại lệ đã quan sát trong dữ liệu.

In [5]:
bad_remodel = X_train['YearRemodAdd'] > X_train['YrSold']
display(pd.concat([
    X_train.loc[bad_remodel, ['YearBuilt', 'YearRemodAdd', 'YrSold']],
    features.loc[bad_remodel, ['HouseAge', 'RemodelAge', 'HasRemodel']],
], axis=1))

,YearBuilt,YearRemodAdd,YrSold,HouseAge,RemodelAge,HasRemodel
523,2007,2008,2007,0,NaN,NaN


## 5 Kiểm tra kết quả bàn giao

Kiểm tra đủ biến, không trùng và giữ nguyên dòng train. Các cột nguồn dùng để tạo cờ
có/không không bị thiếu trong train hiện tại. Phần này không kiểm thử dữ liệu giả định.

In [6]:
assert len(groups) == 79 and groups['bien'].is_unique
assert set(groups['bien']) == set(X_train.columns)
assert len(proposals) == 21 and len(core) == 9
assert features.shape == (len(X_train), 9)
assert features.index.equals(X_train.index)
flag_sources = [
    'GarageCars', 'GarageArea', 'TotalBsmtSF', 'Fireplaces',
    'YearBuilt', 'YearRemodAdd', 'YrSold',
]
assert X_train[flag_sources].notna().all().all()
print('Đạt: bảng 79 biến, 21 đề xuất và 9 đặc trưng minh họa trên train.')

Đạt: bảng 79 biến, 21 đề xuất và 9 đặc trưng minh họa trên train.


## 6 Bàn giao cho nhóm

Dùng cùng các fold CV để so sánh biến gốc với biến gốc cộng 9 đặc trưng cốt lõi.
Các bước impute, encode và scale cần nằm trong Pipeline; giữ holdout cho đánh giá cuối.
12 ý tưởng thử thêm vẫn nằm trong tài liệu và bảng CSV, chưa được tính trong notebook này.

Notebook nằm trong `drafts/` theo cấu trúc của nhóm. Khi đóng gói nộp, dùng tài liệu và CSV
trong `01_Report/HoangAnh/`; công thức được nhóm chọn sẽ ghép vào hai notebook chính.

Tái tạo bảng thống kê 9 đặc trưng, từ `03_SourceCode` chạy:
```cmd
.venv\Scripts\python.exe -m src.feature_proposal
```